# Demo: Retrieval y Sistema Agéntico

Este notebook demuestra las diferentes estrategias de retrieval y el sistema agéntico.

In [ ]:
import sys
sys.path.append('..')

from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.retrieval.vector_retriever import VectorRetriever, HybridRetriever
from graphrag.retrieval.fulltext_retriever import FullTextRetriever
from graphrag.retrieval.text2cypher import Text2CypherRetriever
from graphrag.agents import AgenticRAG

## 1. Inicializar conexión

In [ ]:
neo4j = Neo4jManager()

## 2. Probar Vector Retrieval

In [ ]:
vector_retriever = VectorRetriever(neo4j)

results = vector_retriever.retrieve("What did Einstein work on?")

print("Vector Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text'][:200]}...")

## 3. Probar Full-text y Hybrid Retrieval

In [ ]:
fulltext_retriever = FullTextRetriever(neo4j)

results = fulltext_retriever.retrieve("Einstein Nobel Prize")

print("Full text Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text'][:200]}...")

In [ ]:
hybrid_retriever = HybridRetriever(neo4j)

results = hybrid_retriever.retrieve("Einstein Nobel Prize")

print("Hybrid Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text'][:200]}...")

## 4. Probar Text2Cypher

In [ ]:
text2cypher = Text2CypherRetriever(neo4j)

# Añadir ejemplos few-shot
text2cypher.add_few_shot_example(
    "What entities are connected to Einstein?",
    "MATCH (e:Entity {name: 'Albert Einstein'})-[r:RELATIONSHIP]-(other) RETURN other.name, r.type"
)

cypher, results = text2cypher.retrieve("How many PERSON entities are there?")

print("Generated Cypher:")
print(cypher)
print("\nResults:")
for result in results:
    print(result)

## 5. Probar Sistema Agéntico

In [ ]:
agentic_rag = AgenticRAG(neo4j)

# Pregunta simple
result = agentic_rag.answer("Where was Einstein born?")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nIterations:", result['final_critique'])

In [ ]:
# Pregunta compleja que requiere agregación
result = agentic_rag.answer("List all locations mentioned in the documents")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nTool used:", result['iterations'][-1]['retrieval']['tool'])
print("\nRouting decision:", result['iterations'][-1]['retrieval']['routing_decision']['reasoning'])

In [ ]:
# Conversación multi-turno
agentic_rag.reset_conversation()

result1 = agentic_rag.answer("What theory did Einstein develop?")
print("Q1:", result1['question'])
print("A1:", result1['answer'])

result2 = agentic_rag.answer("When was it published?")
print("\nQ2:", result2['question'])
print("A2:", result2['answer'])

In [ ]:
neo4j.close()